<a href="https://colab.research.google.com/github/naubergois/curso-yolo-colab/blob/main/notebooks/04-treino-com-dataset-proprio.ipynb" target="_blank"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Abrir no Colab"/></a>

# 4. Treinar o YOLO com um dataset

O modelo pronto conhece 80 classes. Para reconhecer algo novo (uma peça da sua fábrica, um tipo de placa, uma praga na folha), você treina com fotos rotuladas. O Ultralytics faz isso com uma chamada só.

Para caber numa aula, o treino usa o `coco8`: oito fotos (quatro de treino, quatro de validação), já no formato certo, que baixam sozinhas. Serve para entender o processo, não para ter um modelo bom. Isso pede centenas de imagens por classe.

> Aqui a GPU faz diferença: **Ambiente de execução → Alterar o tipo de ambiente → T4 GPU**.

In [ ]:
!pip install -q ultralytics

## Como é um dataset YOLO

Um arquivo `.yaml` aponta as pastas de treino e validação e lista as classes. Para cada imagem existe um `.txt` com uma linha por objeto: `classe x_centro y_centro largura altura`, tudo de 0 a 1.

In [ ]:
from pathlib import Path

import matplotlib.pyplot as plt
import pandas as pd
from ultralytics import YOLO
from ultralytics.utils import DATASETS_DIR

model = YOLO("yolov8n.pt")

## Treinar

Começamos dos pesos já treinados (`yolov8n.pt`). Isso se chama *transfer learning*: o modelo já sabe ver bordas e formas, só ajusta o final. Cinco épocas com imagens de 320 px levam poucos minutos.

In [ ]:
treino = model.train(
    data="coco8.yaml",
    epochs=5,
    imgsz=320,
    batch=8,
    project="runs",
    name="coco8",
    exist_ok=True,
    plots=False,
    verbose=False,
)
pasta = Path(treino.save_dir)
print("Resultados em", pasta)

## O que o arquivo de rótulos tem

In [ ]:
rotulo = sorted((DATASETS_DIR / "coco8" / "labels" / "train").glob("*.txt"))[0]
print(rotulo.name)
print(rotulo.read_text())

## Acompanhar as perdas

O `results.csv` guarda as métricas de cada época. Num dataset de verdade a perda (`loss`) cai época após época e o `mAP50`, que mede acerto na validação, sobe. Aqui são quatro fotos de treino e cinco épocas, então as curvas oscilam. O que importa é ver onde cada número mora.

In [ ]:
df = pd.read_csv(pasta / "results.csv")
df.columns = df.columns.str.strip()
print(df[["epoch", "train/box_loss", "train/cls_loss", "metrics/mAP50(B)"]].to_string(index=False))

plt.figure(figsize=(8, 3))
plt.plot(df["epoch"], df["train/box_loss"], label="box_loss")
plt.plot(df["epoch"], df["train/cls_loss"], label="cls_loss")
plt.xlabel("época")
plt.legend()
plt.grid(True)
plt.show()

## Usar o modelo treinado

O melhor peso fica em `weights/best.pt`. Ele carrega igual ao modelo original.

In [ ]:
melhor = YOLO(pasta / "weights" / "best.pt")
r = melhor("https://ultralytics.com/images/bus.jpg", verbose=False)[0]
plt.figure(figsize=(6, 8))
plt.imshow(r.plot()[:, :, ::-1])
plt.axis("off")
plt.show()
print(len(r.boxes), "objetos detectados")

## Para treinar com as suas fotos

1. Rotule as imagens no [Roboflow](https://roboflow.com) ou no [CVAT](https://www.cvat.ai) e exporte em **YOLOv8**.
2. Suba o `.zip` no Colab e descompacte.
3. Troque `data="coco8.yaml"` pelo `data.yaml` do seu dataset e aumente `epochs` para 50 ou mais.

O processo é o mesmo desta aula. Muda o tamanho do dataset e o tempo de treino.